# LeetCode #1223: Dice Roll Simulation

https://leetcode.com/problems/dice-roll-simulation/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Recursion + Memo)** | $O(n \times 6 \times \max(rollMax))$ | $O(n \times 6 \times \max(rollMax))$ |
| **Optimal: DP with Subtraction ★** | $O(n \times 6 \times \max(rollMax))$ | $O(6 \times \max(rollMax))$ |

---

## Understanding the Methods

### Brute Force (Recursion + Memo)
Use top-down DP: `memo[step][face][streak]` = number of valid sequences. Each state tracks which roll step, the last face rolled, and the current consecutive count. Memoization avoids recomputation but uses $O(n \times 6 \times \max(rollMax))$ space.

### Optimal: DP with Subtraction ★
`dp[i][f]` = number of valid sequences of length $i$ ending with face $f$. When extending, the total for face $f$ after $i$ rolls is the sum of all `dp[i-1][g]` across all faces $g$ (i.e., the grand total), minus sequences that would exceed `rollMax[f]`. The "subtract" trick avoids tracking streak in the state and allows rolling space down to $O(6 \times \max(rollMax))$.

**Constraints:**
* $1 \leq n \leq 5000$
* `rollMax.length == 6`
* $1 \leq rollMax[i] \leq 15$

## Solutions

### C#

In [ ]:
public class Solution {
    const int MOD = 1_000_000_007;
    public int DieSimulator(int n, int[] rollMax) {
        // dp[f][c]: ways to have the last roll be face f repeated c consecutive times
        // Indices: f in [0,5], c in [1, rollMax[f]]
        int maxR = 0; foreach (int r in rollMax) if (r > maxR) maxR = r;
        var dp = new long[6, maxR + 1];
        // Initialise: after 1 roll, each face has exactly 1 way with streak=1
        for (int f = 0; f < 6; f++) dp[f, 1] = 1;
        for (int i = 2; i <= n; i++) {
            var next = new long[6, maxR + 1];
            // Total valid sequences of length i-1
            long total = 0;
            for (int f = 0; f < 6; f++)
                for (int c = 1; c <= rollMax[f]; c++)
                    total = (total + dp[f, c]) % MOD;
            for (int f = 0; f < 6; f++) {
                // Sequences ending in f with streak 1 = total minus those already maxed out on f
                // "shift" existing f-streaks: streak c → c+1 (if still within rollMax[f])
                for (int c = rollMax[f]; c >= 2; c--)
                    next[f, c] = dp[f, c - 1];
                // New streak of 1: contributed by all other faces' totals
                long fTotal = 0;
                for (int c2 = 1; c2 <= rollMax[f]; c2++) fTotal = (fTotal + dp[f, c2]) % MOD;
                next[f, 1] = (total - fTotal % MOD + MOD) % MOD;
            }
            dp = next;
        }
        long ans = 0;
        for (int f = 0; f < 6; f++)
            for (int c = 1; c <= rollMax[f]; c++)
                ans = (ans + dp[f, c]) % MOD;
        return (int)ans;
    }
}

### Python

In [ ]:
class Solution:
    def die_simulator(self, n: int, roll_max: list[int]) -> int:
        MOD = 10**9 + 7
        max_r = max(roll_max)
        # dp[f][c]: ways to have the last roll be face f repeated c consecutive times
        dp = [[0] * (max_r + 1) for _ in range(6)]
        for f in range(6): dp[f][1] = 1  # after 1 roll each face has 1 way streak=1
        for _ in range(2, n + 1):
            nxt = [[0] * (max_r + 1) for _ in range(6)]
            total = sum(dp[f][c] for f in range(6) for c in range(1, roll_max[f]+1)) % MOD
            for f in range(6):
                f_total = sum(dp[f][c] for c in range(1, roll_max[f]+1)) % MOD
                # Sequences ending with a fresh f (streak=1) = total minus existing f-only chains
                nxt[f][1] = (total - f_total + MOD) % MOD
                # Shift existing streaks: c-1 consecutive f's become c consecutive
                for c in range(2, roll_max[f] + 1):
                    nxt[f][c] = dp[f][c - 1]
            dp = nxt
        return sum(dp[f][c] for f in range(6) for c in range(1, roll_max[f]+1)) % MOD

### Go

In [ ]:
func dieSimulator(n int, rollMax []int) int {
    const MOD = 1_000_000_007
    maxR := 0
    for _, r := range rollMax { if r > maxR { maxR = r } }
    // dp[f][c]: ways to have the last roll be face f repeated c consecutive times
    dp := make([][]int64, 6)
    for f := range dp { dp[f] = make([]int64, maxR+1) }
    for f := 0; f < 6; f++ { dp[f][1] = 1 }
    for i := 2; i <= n; i++ {
        next := make([][]int64, 6)
        for f := range next { next[f] = make([]int64, maxR+1) }
        var total int64
        for f := 0; f < 6; f++ {
            for c := 1; c <= rollMax[f]; c++ { total = (total + dp[f][c]) % MOD }
        }
        for f := 0; f < 6; f++ {
            var fTotal int64
            for c := 1; c <= rollMax[f]; c++ { fTotal = (fTotal + dp[f][c]) % MOD }
            // Fresh streak of 1 for face f: total minus sequences already using only face f
            next[f][1] = (total - fTotal + MOD) % MOD
            // Shift streaks: c-1 consecutive f's → c consecutive
            for c := rollMax[f]; c >= 2; c-- { next[f][c] = dp[f][c-1] }
        }
        dp = next
    }
    var ans int64
    for f := 0; f < 6; f++ {
        for c := 1; c <= rollMax[f]; c++ { ans = (ans + dp[f][c]) % MOD }
    }
    return int(ans)
}

### Rust

In [ ]:
impl Solution {
    pub fn die_simulator(n: i32, roll_max: Vec<i32>) -> i32 {
        const MOD: i64 = 1_000_000_007;
        let max_r = *roll_max.iter().max().unwrap() as usize;
        // dp[f][c]: ways to have the last roll be face f repeated c consecutive times
        let mut dp = vec![vec![0i64; max_r + 1]; 6];
        for f in 0..6 { dp[f][1] = 1; }
        for _ in 2..=n {
            let mut next = vec![vec![0i64; max_r + 1]; 6];
            let total: i64 = (0..6).flat_map(|f| (1..=roll_max[f] as usize).map(move |c| (f,c)))
                .map(|(f,c)| dp[f][c]).sum::<i64>() % MOD;
            for f in 0..6 {
                let rm = roll_max[f] as usize;
                let f_total: i64 = (1..=rm).map(|c| dp[f][c]).sum::<i64>() % MOD;
                // Fresh streak-1 for face f
                next[f][1] = (total - f_total + MOD) % MOD;
                // Shift existing streaks upward
                for c in 2..=rm { next[f][c] = dp[f][c-1]; }
            }
            dp = next;
        }
        let ans: i64 = (0..6).flat_map(|f| (1..=roll_max[f] as usize).map(move |c| (f,c)))
            .map(|(f,c)| dp[f][c]).sum::<i64>() % MOD;
        ans as i32
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n = 2`, `rollMax = [1,1,2,2,2,3]`
After 2 rolls, most face-pairs are allowed. The DP counts: valid sequences where no face exceeds its rollMax consecutively. Answer: **34**.

### 2. Slightly Complex
**Input:** `n = 2`, `rollMax = [1,1,1,1,1,1]`
No face can repeat. Every sequence of two different faces is valid: $6 \times 5 = 30$ ways. Answer: **30**.

### 3. Edge Case: Time Factor
**Input:** `n = 5000`, `rollMax = [15,15,15,15,15,15]`.
Each of the 5000 iterations processes $6 \times 15 = 90$ states. Total operations $\approx 5000 \times 6 \times (15 + 6) \approx 630{,}000$ — well within $O(n \times 6 \times \max(rollMax))$.

### 4. Edge Case: Space Factor
**Input:** `rollMax = [15,15,15,15,15,15]`.
Rolling DP arrays hold $6 \times 15 = 90$ values at a time — $O(6 \times \max(rollMax))$ space regardless of $n = 5000$.

### 5. Almost-Impossible but Plausible
**Input:** `n = 5000`, `rollMax = [1,1,1,1,1,1]`.
Every roll must differ from the previous. At each step the total exactly equals $5 \times$ (previous total) $/ 6 \times$ … modular arithmetic keeps values bounded. The streak dimension collapses to just $c=1$ for all faces; the DP degenerates to a simple branching factor of 5.